# 02 — Model Training & Evaluation

**Project**: Auto Tagging Ticket Support System  
**Model**: TF-IDF + Logistic Regression (multi-class)

## Steps
1. Load & split data (80/20)
2. Build sklearn Pipeline (TF-IDF + LogReg)
3. Train model
4. Evaluate: Accuracy, F1 (weighted), Confusion Matrix, Classification Report
5. Save model artifact (`models/model.pkl`)

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

from src.preprocessing import (
    load_and_prepare_data,
    preprocess_series,
    build_tfidf_vectorizer
)

print('Libraries loaded.')

## 1. Load Data

In [ ]:
# Load processed data (from EDA notebook) or raw data
import os

PROCESSED_PATH = '../data/processed/tickets_clean.csv'
RAW_PATH = '../data/raw/customer_support_tickets.csv'

if os.path.exists(PROCESSED_PATH):
    df = pd.read_csv(PROCESSED_PATH)
    print(f'Loaded processed data: {PROCESSED_PATH}')
else:
    df = load_and_prepare_data(RAW_PATH)
    df['text_clean'] = preprocess_series(df['text'])
    print(f'Loaded and preprocessed raw data: {RAW_PATH}')

print(f'Shape: {df.shape}')
print(f'Labels: {sorted(df["label"].unique())}')
df.head()

## 2. Train/Test Split (80/20)

In [ ]:
TEXT_COL = 'text_clean' if 'text_clean' in df.columns else 'text'

X_train, X_test, y_train, y_test = train_test_split(
    df[TEXT_COL],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(f'Train set : {len(X_train):,} samples')
print(f'Test set  : {len(X_test):,} samples')
print()
print('Train label distribution:')
print(y_train.value_counts())

## 3. Build sklearn Pipeline

In [ ]:
pipeline = Pipeline([
    ('tfidf', build_tfidf_vectorizer(max_features=10_000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(
        solver='lbfgs',
        multi_class='auto',
        max_iter=1000,
        C=1.0,
        random_state=42,
        n_jobs=-1
    ))
])

print('Pipeline built:')
print(pipeline)

## 4. Train Model

In [ ]:
print('Training model...')
pipeline.fit(X_train, y_train)
print('Training complete.')

## 5. Evaluate Model

In [ ]:
y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')

print('=' * 55)
print(f'  Accuracy         : {accuracy:.4f}  ({accuracy:.2%})')
print(f'  F1-score (weighted): {f1:.4f}  ({f1:.2%})')
print('=' * 55)

target_f1 = 0.75
if f1 >= target_f1:
    print(f'  ✓ F1-score meets target of {target_f1:.0%}')
else:
    print(f'  ⚠ F1-score ({f1:.2%}) is below target of {target_f1:.0%}')

In [ ]:
# Full classification report
print(classification_report(y_test, y_pred))

### 5a. Per-class F1 Visualization

In [ ]:
report = classification_report(y_test, y_pred, output_dict=True)
labels = sorted(df['label'].unique())

class_metrics = []
for label in labels:
    if label in report:
        class_metrics.append({
            'Category': label,
            'Precision': report[label]['precision'],
            'Recall': report[label]['recall'],
            'F1-Score': report[label]['f1-score'],
            'Support': int(report[label]['support'])
        })

class_df = pd.DataFrame(class_metrics)

fig = px.bar(
    class_df,
    x='F1-Score', y='Category',
    orientation='h',
    color='F1-Score',
    color_continuous_scale='RdYlGn',
    range_color=[0, 1],
    title='F1-Score per Category'
)
fig.add_vline(x=0.75, line_dash='dash', line_color='red', annotation_text='Target: 0.75')
fig.update_layout(height=max(300, len(class_df) * 40))
fig.show()

class_df

### 5b. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=labels)

# Normalized confusion matrix
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)

fig = px.imshow(
    cm_norm,
    x=labels, y=labels,
    color_continuous_scale='Blues',
    title='Confusion Matrix (Normalized)',
    labels={'x': 'Predicted', 'y': 'Actual', 'color': 'Rate'},
    text_auto='.2f'
)
fig.update_layout(height=max(400, len(labels) * 50))
fig.show()

### 5c. Cross-Validation (optional — runs longer)

In [ ]:
# Uncomment to run 5-fold cross-validation
# cv_scores = cross_val_score(pipeline, df[TEXT_COL], df['label'], cv=5, scoring='f1_weighted', n_jobs=-1)
# print(f'CV F1 (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print('Cross-validation skipped. Uncomment above to run.')

## 6. Save Model Artifacts

In [ ]:
import os

os.makedirs('../models', exist_ok=True)
MODEL_PATH = '../models/model.pkl'
METRICS_PATH = '../models/metrics.json'

# Save pipeline
joblib.dump(pipeline, MODEL_PATH)
print(f'Model saved → {MODEL_PATH}')

# Save metrics
metrics = {
    'accuracy': round(accuracy, 4),
    'f1_weighted': round(f1, 4),
    'classification_report': report,
    'confusion_matrix': cm.tolist(),
    'labels': labels,
    'train_size': len(X_train),
    'test_size': len(X_test),
    'num_classes': df['label'].nunique(),
    'model_type': 'TF-IDF + Logistic Regression'
}

with open(METRICS_PATH, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Metrics saved → {METRICS_PATH}')

## 7. Quick Inference Test

In [ ]:
from src.predict import predict_tag, predict_batch

test_tickets = [
    'My internet connection keeps dropping every few minutes.',
    'I was charged twice for the same order. Please refund the extra charge.',
    'How do I reset my password? I cannot log into my account.',
    'The package was supposed to arrive yesterday but it is still in transit.',
    'I want to cancel my subscription effective immediately.',
]

print('=' * 60)
print('Inference Test')
print('=' * 60)

for ticket in test_tickets:
    result = predict_tag(ticket)
    print(f'Ticket     : {ticket}')
    print(f'  Tag      : {result["tag"]}')
    print(f'  Confidence: {result["confidence"]:.2%}')
    print()

## 8. Model Training Summary

| Metric | Value |
|--------|-------|
| Model | TF-IDF (10k features, 1-2 ngrams) + LogisticRegression (lbfgs) |
| Train/Test Split | 80% / 20% |
| Accuracy | *(fill after running)* |
| F1-score (weighted) | *(fill after running)* |
| Meets Target (F1 ≥ 0.75) | *(fill after running)* |

**Next Step**: Run `streamlit run app/main.py` to launch the dashboard.